### 1. Data Ingestion: Steam Games CSV
Reading the static historical dataset. Applied `escape` and `multiLine` options to handle complex game descriptions and prevent column shifting.

In [0]:
file_path = "/Volumes/workspace/default/raw_data/games.csv"

steam_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .load(file_path)

In [0]:
steam_df.printSchema()
print(f"Total rows ingested: {steam_df.count()}")
print(f"Total columns: {len(steam_df.columns)}")

In [0]:
display(steam_df.limit(10))

In [0]:
from pyspark.sql.functions import col, sum as _sum

display(
    steam_df.select([
        _sum(col(c).isNull().cast("int")).alias(c) for c in steam_df.columns
    ])
)

### 2. External API Integration
Fetching live current player counts for specific popular titles from the external Steam API to enrich our dataset.

In [0]:
import requests
from pyspark.sql.types import StructType, StructField, IntegerType

# We pick a few known AppIDs (e.g., CS2, Dota 2, PUBG) to demonstrate API integration
target_app_ids = [730, 570, 578080]
api_records = []

for app_id in target_app_ids:
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            player_count = data.get("response", {}).get("player_count", 0)
            api_records.append((app_id, player_count))
    except Exception as e:
        print(f"Failed to fetch data for {app_id}: {e}")

# Create DataFrame from API results
api_schema = StructType([
    StructField("AppID", IntegerType(), True),
    StructField("CurrentPlayers", IntegerType(), True)
])

api_df = spark.createDataFrame(api_records, api_schema)
display(api_df)

In [0]:
from pyspark.sql.functions import col

games_names = steam_df.select("AppID", "Name").where(col("AppID").isin(target_app_ids))
display(games_names)